# Veritas-R1 — Stage 1.5: Synthetic-paraphrase augmentation

Optional. Mirrors Meta's Llama 3.1 §3 "synthesise-then-filter" pipeline at small scale. Use the SFT'd Stage-1 model itself as a paraphraser to rewrite ~25% of training prompts (targets kept intact). Adds extra training examples without paying for new annotation.

**Inputs**: Stage-1 SFT adapter from `veritas-stage1-train.ipynb`.
**Outputs**: `/kaggle/working/augmented/sft_train_augmented.parquet`.

In [ ]:
%pip install --quiet --upgrade "transformers>=4.46,<5" "peft>=0.13" "datasets>=3.0" "bitsandbytes>=0.44"

In [ ]:
import json, random, gc
from pathlib import Path
import pandas as pd
import torch
from datasets import load_dataset

BASE_MODEL = "Qwen/Qwen3-1.7B-Instruct"
OUT_ROOT = Path("/kaggle/working")
STAGE1_ADAPTER = OUT_ROOT / "adapters" / "seed_17" / "adapter"
AUG_DIR = OUT_ROOT / "augmented"
AUG_DIR.mkdir(parents=True, exist_ok=True)

TARGET_AUGS = 60_000
MAX_NEW = 256
device = "cuda" if torch.cuda.is_available() else "cpu"

## 1. Pull seed prompts from the workspace-relevant pools

In [ ]:
def collect_seeds(target=TARGET_AUGS):
    pool = []
    sources = [
        ("HuggingFaceTB/smol-talk-2", "messages"),
        ("allenai/tulu-3-sft-mixture", "messages"),
        ("open-thoughts/OpenThoughts-114k", "conversations"),
    ]
    for hf, key in sources:
        ds = load_dataset(hf, split="train", streaming=True)
        for row in ds:
            content = row.get(key)
            user_text = target_text = None
            if not isinstance(content, list) or not content: continue
            if key == "messages":
                for m in content:
                    if m.get("role") == "user" and m.get("content") and not user_text:
                        user_text = m["content"]
                    elif m.get("role") == "assistant" and m.get("content") and user_text:
                        target_text = m["content"]; break
            else:
                for t in content:
                    if t.get("from") in {"user","human"} and not user_text:
                        user_text = t.get("value")
                    elif t.get("from") in {"assistant","gpt"} and user_text:
                        target_text = t.get("value"); break
            if user_text and target_text:
                pool.append({"original_prompt": str(user_text)[:2000], "target": str(target_text)[:6000]})
                if len(pool) >= target * 2: break
        if len(pool) >= target * 2: break
    rng = random.Random(11); rng.shuffle(pool)
    return pool[:target * 2]

SEEDS = collect_seeds()
print(f"seed pool: {len(SEEDS)}")

## 2. Load the SFT'd Stage-1 model

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.bfloat16)
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token
base = AutoModelForCausalLM.from_pretrained(BASE_MODEL, quantization_config=bnb, torch_dtype=torch.bfloat16, attn_implementation="flash_attention_2" if device=="cuda" else "eager")
model = PeftModel.from_pretrained(base, str(STAGE1_ADAPTER))
model.eval()
print("loaded paraphraser")

## 3. Generate paraphrases + filter

In [ ]:
PARAPHRASE_TEMPLATE = (
    "Rewrite the following user request so that it asks for the same thing in different words. "
    "Keep the meaning, level of detail, and tone identical. Output ONLY the rewrite — no preamble, no quotes.\n\n"
    "Original:\n{prompt}\n\nRewrite:"
)

@torch.no_grad()
def paraphrase(prompt):
    msgs = [
        {"role": "system", "content": "You are a precise paraphrasing assistant."},
        {"role": "user", "content": PARAPHRASE_TEMPLATE.format(prompt=prompt)},
    ]
    input_ids = tokenizer.apply_chat_template(msgs, return_tensors="pt", add_generation_prompt=True).to(device)
    out = model.generate(input_ids, max_new_tokens=MAX_NEW, do_sample=True, temperature=0.8, top_p=0.9, pad_token_id=tokenizer.pad_token_id)
    return tokenizer.decode(out[0, input_ids.shape[1]:], skip_special_tokens=True).strip()

def filter_paraphrase(original, rewrite):
    if not rewrite or len(rewrite) < 16: return False
    if len(rewrite) > 2 * len(original) + 200: return False
    if rewrite.strip().lower() == original.strip().lower(): return False
    if "original:" in rewrite.lower() or "rewrite:" in rewrite.lower(): return False
    return True

rows = []
for i, seed in enumerate(SEEDS):
    if len(rows) >= TARGET_AUGS: break
    rewrite = paraphrase(seed["original_prompt"])
    if not filter_paraphrase(seed["original_prompt"], rewrite): continue
    rows.append({"original_prompt": seed["original_prompt"], "paraphrased_prompt": rewrite, "target": seed["target"]})
    if len(rows) % 500 == 0:
        print(f"  paraphrased={len(rows)} (seen={i+1})")

df = pd.DataFrame(rows)
out_path = AUG_DIR / "sft_train_augmented.parquet"
df.to_parquet(out_path, index=False)
print(f"saved {len(df)} paraphrased examples → {out_path}")

## 4. Use augmented data in next training round

In `veritas-stage1-train.ipynb`, after the curriculum is built:

```python
df = pd.read_parquet("/kaggle/working/augmented/sft_train_augmented.parquet")
for _, r in df.iterrows():
    train_pool.append({"messages": [
        {"role": "system",    "content": SYSTEM_DEFAULT},
        {"role": "user",      "content": r.paraphrased_prompt},
        {"role": "assistant", "content": r.target},
    ]})
```

Then re-tokenise + retrain.